In [1]:
import os, re, glob, logging, unicodedata
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# ── PATH (relatif dari notebooks/) ──────────────────────────────
BASE        = Path('..').resolve()          # root projek-cbr-hukum/
INPUT_DIR   = BASE / 'data' / 'raw'         # ← letakkan PDF/TXT di sini
OUTPUT_DIR  = BASE / 'data' / 'raw'
LOG_DIR     = BASE / 'logs'

for d in [INPUT_DIR, OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Logger ───────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_DIR / 'cleaning.log', encoding='utf-8'),
    ]
)
log = logging.getLogger('step1')
print('Setup selesai ✅')

Setup selesai ✅


In [2]:
# ── Pola header/footer khas Direktori MA ────────────────────────
HEADER_PATTERNS = [
    r'direktori putusan mahkamah agung republik indonesia',
    r'mahkamah agung republik indonesia',
    r'putusan\.mahkamahagung\.go\.id',
    r'halaman\s*\d+\s*dari\s*\d+\s*halaman',
    r'catatan\s+catatan',
    r'-\s*\d+\s*-',
    r'={5,}', r'-{5,}', r'\f',
]

def extract_pdf(path: Path) -> str:
    try:
        from pdfminer.high_level import extract_text
        return extract_text(str(path)) or ''
    except Exception as e:
        log.error(f'Gagal ekstrak PDF {path.name}: {e}')
        return ''

def extract_txt(path: Path) -> str:
    try:
        return path.read_text(encoding='utf-8', errors='replace')
    except Exception as e:
        log.error(f'Gagal baca TXT {path.name}: {e}')
        return ''

def clean_text(raw: str) -> str:
    """Normalisasi unicode → lowercase → hapus header/footer → normalisasi spasi."""
    text = unicodedata.normalize('NFKC', raw).lower()
    for pat in HEADER_PATTERNS:
        text = re.sub(pat, ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', ' ', text)  # karakter kontrol
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def validate(text: str) -> dict:
    """Validasi kualitas: min 500 karakter + 3 elemen kunci putusan."""
    has_putusan  = bool(re.search(r'putusan|mengadili', text))
    has_perkara  = bool(re.search(r'\d+/pid|\d+/pdt|nomor\s+\d+', text))
    has_dakwaan  = bool(re.search(r'terdakwa|dakwaan|penuntut', text))
    score = (has_putusan + has_perkara + has_dakwaan) / 3.0
    return {
        'valid'        : len(text) >= 500 and score >= 0.33,
        'char_count'   : len(text),
        'word_count'   : len(text.split()),
        'quality_score': round(score, 2),
    }

print('Fungsi siap ✅')

Fungsi siap ✅


In [5]:
import sys
!{sys.executable} -m pip install pdfminer.six

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.6 MB 626.2 kB/s eta 0:00:10
   --- ------------------------------------ 0.5/6.6 MB 626.2 kB/s eta 0:00:10
   ---- ----------------------------------- 0.8/6.6 MB 567.4 kB/s eta 0:00:11
   ---- ----------------------------------- 0.8/6.6 MB 567.4 kB/s eta 0:00:11
   ------ --------------------------------- 1.0/6.6 MB 571.6 kB/s eta 0:00:10
   ------ --------------------------------- 1.0/6.6 MB 571.6 kB/s eta 0:00:10
   ------- -------------------------------- 1.3/6.6 MB 577.4 kB/s eta 0:00:10
   ------- -------------------------------- 1.3/6.6 MB 577.4 kB/s eta 0:00:10
   --------- ------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ── Jalankan Pipeline ────────────────────────────────────────────
files = sorted(list(INPUT_DIR.rglob('*.pdf')) + list(INPUT_DIR.rglob('*.txt')))
# Hapus file output yang sudah bernama case_XXX.txt agar tidak terbaca ulang
files = [f for f in files if not re.match(r'case_\d+\.txt', f.name)]

print(f'Ditemukan {len(files)} file input')

ok, skip = 0, 0
for idx, fpath in enumerate(tqdm(files, desc='Preprocessing'), start=1):
    case_id = f'case_{idx:03d}'
    raw     = extract_pdf(fpath) if fpath.suffix == '.pdf' else extract_txt(fpath)
    cleaned = clean_text(raw)
    val     = validate(cleaned)

    if not val['valid']:
        log.warning(f'SKIP {case_id} ({fpath.name}) | chars={val["char_count"]} | q={val["quality_score"]}')
        skip += 1
        continue

    out = OUTPUT_DIR / f'{case_id}.txt'
    out.write_text(cleaned, encoding='utf-8')
    log.info(f'OK   {case_id} | {val["word_count"]} kata | q={val["quality_score"]}')
    ok += 1

print(f'\n✅ Selesai: {ok} berhasil, {skip} dilewati')
print(f'📁 Output : {OUTPUT_DIR}')
if ok < 30:
    print(f'⚠️  Dokumen valid hanya {ok} — syarat minimal 30 belum terpenuhi!')

Ditemukan 50 file input


Preprocessing: 100%|██████████| 50/50 [01:05<00:00,  1.31s/it]


✅ Selesai: 50 berhasil, 0 dilewati
📁 Output : C:\Users\endo1\Documents\SEMESTER 6\PK\tugas 3\Penalaran-Komputer_SubCpmk3\data\raw


In [4]:
# ── Ringkasan ────────────────────────────────────────────────────
hasil = sorted(OUTPUT_DIR.glob('case_*.txt'))
print(f'Total file di data/raw/ : {len(hasil)}')
if hasil:
    sample = hasil[0].read_text(encoding='utf-8')
    print(f'\nPreview {hasil[0].name}:')
    print('-' * 60)
    print(sample[:400], '...')

Total file di data/raw/ : 50

Preview case_001.txt:
------------------------------------------------------------
disclaimer
kepaniteraan berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen mahkamah agung untuk pelayanan publik, transparansi dan akuntabilitas
pelaksanaan fungsi peradilan. namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari w ...


In [5]:
import re, sys, json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

BASE          = Path('..').resolve()
RAW_DIR       = BASE / 'data' / 'raw'
PROCESSED_DIR = BASE / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_files = sorted(RAW_DIR.glob('case_*.txt'))
print(f'File ditemukan: {len(raw_files)}')

File ditemukan: 50


In [6]:
# ════════════════════════════════════════════════════════════
# FUNGSI EKSTRAKSI METADATA
# ════════════════════════════════════════════════════════════

BULAN = {
    'januari':'01','februari':'02','maret':'03','april':'04',
    'mei':'05','juni':'06','juli':'07','agustus':'08',
    'september':'09','oktober':'10','november':'11','desember':'12'
}

def get_nomor_perkara(t):
    for pat in [
        r'\d{1,4}/pid\.?sus[\w./\-]*',
        r'\d{1,4}/pid[\w./\-]*',
        r'\d{1,4}/pdt[\w./\-]*',
        r'nomor\s*[:\-]?\s*([\d/a-z\.\-\s]{8,60})',
    ]:
        m = re.search(pat, t[:3000], re.IGNORECASE)
        if m:
            val = m.group(0) if m.lastindex is None else m.group(1)
            return re.sub(r'\s+', ' ', val.strip()).upper()[:80]
    return 'N/A'

def get_tanggal(t):
    m = re.search(
        r'(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|'
        r'september|oktober|november|desember)\s+(20\d{2}|19\d{2})',
        t[:5000], re.IGNORECASE
    )
    if m:
        return f"{m.group(3)}-{BULAN.get(m.group(2).lower(),'00')}-{m.group(1).zfill(2)}"
    return 'N/A'

def get_jenis_perkara(t):
    if any(k in t for k in ['narkotika','psikotropika','sabu','ganja','kokain']):
        return 'Pidana Khusus Narkotika'
    if any(k in t for k in ['korupsi','gratifikasi','suap']):
        return 'Pidana Khusus Korupsi'
    if 'wanprestasi' in t or 'perbuatan melawan hukum' in t:
        return 'Perdata'
    if 'pid.sus' in t: return 'Pidana Khusus'
    if '/pid' in t:    return 'Pidana Umum'
    return 'Tidak Terdeteksi'

def get_pasal(t):
    pats = [
        r'pasal\s+\d+\s*(?:ayat\s*\(\d+\)\s*)?(?:jo\.?\s*pasal\s+\d+[^;\n]{0,40})?'
        r'(?:uu\s+(?:nomor|no\.?)\s+\d+\s+tahun\s+\d+[^;\n]{0,40})?',
        r'pasal\s+\d+\s+kuhp(?:er)?',
    ]
    found = []
    for p in pats:
        found += re.findall(p, t, re.IGNORECASE)
    unique = list(dict.fromkeys(re.sub(r'\s+', ' ', x.strip()) for x in found))
    return '; '.join(unique[:5]) if unique else 'N/A'

def get_pihak(t):
    td, jp = 'N/A', 'N/A'
    m1 = re.search(r'(?:terdakwa)\s*[:\-]?\s*([A-Z][A-Za-z\s\.]{3,50})(?:[,;\n]|bin|binti|berumur)', t[:5000])
    if m1: td = m1.group(1).strip()
    m2 = re.search(r'(?:penuntut\s+umum|jaksa)\s*[:\-]?\s*([A-Z][A-Za-z\s\.]{3,50})(?:[,;\n])', t[:5000])
    if m2: jp = m2.group(1).strip()
    return td, jp

def get_vonis(t):
    pats = [
        r'pidana\s+penjara\s+selama\s+\d+[^.;]{0,60}',
        r'\d+\s*(?:tahun|bulan)\s*(?:penjara|kurungan)',
        r'denda\s+rp[^.;]{0,60}',
        r'(?:bebas|tidak\s+terbukti|lepas\s+dari\s+tuntutan)',
    ]
    found = []
    for p in pats:
        found += re.findall(p, t, re.IGNORECASE)[:2]
    return '; '.join(found[:3]) if found else 'N/A'

def get_amar(t):
    m = re.search(
        r'(?:mengadili|m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i|amar\s+putusan)'
        r'[\s:]*(.{100,1200}?)(?:ditetapkan|demikian|[A-Z]{5,}\s*:|\Z)',
        t, re.DOTALL | re.IGNORECASE
    )
    return re.sub(r'\s+', ' ', m.group(1)).strip()[:700] if m else 'N/A'

def get_fakta(t):
    m = re.search(
        r'(?:dakwaan|tentang\s+duduknya\s+perkara|menimbang)'
        r'[\s:]*(.{200,1200}?)(?:menimbang|tuntutan|\n\n\n|\Z)',
        t, re.DOTALL | re.IGNORECASE
    )
    if m: return re.sub(r'\s+', ' ', m.group(1)).strip()[:600]
    return re.sub(r'\s+', ' ', t[len(t)//4 : len(t)//4 + 600])

def assign_label(vonis):
    v = vonis.lower()
    if any(k in v for k in ['bebas','tidak terbukti','lepas']): return 'bebas'
    m = re.search(r'(\d+)\s*tahun', v)
    if m:
        y = int(m.group(1))
        return 'ringan' if y <= 4 else ('sedang' if y <= 10 else 'berat')
    return 'tidak_diketahui'

print('Fungsi ekstraksi siap ✅')

Fungsi ekstraksi siap ✅


In [8]:
# ════════════════════════════════════════════════════════════
# PROSES SEMUA KASUS
# ════════════════════════════════════════════════════════════
records = []

for fpath in tqdm(raw_files, desc='Ekstraksi metadata'):
    text = fpath.read_text(encoding='utf-8')
    td, jp = get_pihak(text)

    records.append({
        'case_id'        : fpath.stem,
        'no_perkara'     : get_nomor_perkara(text),
        'tanggal'        : get_tanggal(text),
        'jenis_perkara'  : get_jenis_perkara(text),
        'terdakwa'       : td,
        'penuntut_umum'  : jp,
        'pasal'          : get_pasal(text),
        'vonis'          : get_vonis(text),
        'amar_putusan'   : get_amar(text),
        'ringkasan_fakta': get_fakta(text),
        'word_count'     : len(text.split()),
        'text_full'      : text,
    })

df = pd.DataFrame(records)
df['label_vonis']   = df['vonis'].apply(assign_label)
df['text_combined'] = (
    df['ringkasan_fakta'].fillna('') + ' ' +
    df['pasal'].fillna('')            + ' ' +
    df['amar_putusan'].fillna('')
)

print(f'Total kasus: {len(df)}')
print('Distribusi label:')
print(df['label_vonis'].value_counts().to_string())

Ekstraksi metadata: 100%|██████████| 50/50 [00:00<00:00, 111.66it/s]

Total kasus: 50
Distribusi label:
label_vonis
tidak_diketahui    40
bebas              10


In [9]:
# ════════════════════════════════════════════════════════════
# SIMPAN OUTPUT
# ════════════════════════════════════════════════════════════
cols_csv = [c for c in df.columns if c != 'text_full']

csv_path  = PROCESSED_DIR / 'cases.csv'
json_path = PROCESSED_DIR / 'cases.json'

df[cols_csv].to_csv(csv_path, index=False, encoding='utf-8')
df.to_json(json_path, orient='records', force_ascii=False, indent=2)

print(f'✅ Tersimpan:')
print(f'   {csv_path}')
print(f'   {json_path}')

preview_cols = ['case_id','no_perkara','tanggal','jenis_perkara','pasal','vonis','label_vonis']
df[preview_cols].head(5)

✅ Tersimpan:
   C:\Users\endo1\Documents\SEMESTER 6\PK\tugas 3\Penalaran-Komputer_SubCpmk3\data\processed\cases.csv
   C:\Users\endo1\Documents\SEMESTER 6\PK\tugas 3\Penalaran-Komputer_SubCpmk3\data\processed\cases.json


,case_id,no_perkara,tanggal,jenis_perkara,pasal,vonis,label_vonis
0,case_001,1006/PID.B/2024/PN,1974-06-05,Pidana Umum,pasal 170 ayat (1); pasal 170ayat (1); pasal 3...,N/A,tidak_diketahui
1,case_002,1055/PID.B/2024/PN,2005-12-03,Pidana Umum,pasal 355 ayat (2); pasal 351 ayat (3); pasal ...,pidana penjara selama 10 (sepuluh)tahun dan 6 ...,bebas
2,case_003,1063/PID.B/2024/PN,1975-08-17,Pidana Umum,pasal 170 ayat (1); pasal 351 ayat (1) jo pasa...,N/A,tidak_diketahui
3,case_004,1080/PID.B/2024/PN,2006-03-18,Pidana Umum,pasal 170 ayat (1); pasal 351 ayat (1) jo. pas...,pidana penjara selama 2 (dua) tahun dikurangis...,bebas
4,case_005,1101/PID.B/2024/PN,1990-12-02,Pidana Umum,pasal 351 ayat (1); pasal 22 ayat (4); pasal 3...,pidana penjara selama 1 (satu) tahun dan 6 (en...,tidak_diketahui
